# 05 — Пакетная обработка через Google Drive

Автономная обработка инвойсов: читает PDF из Google Drive, обрабатывает через Gemini API, сохраняет результаты JSON обратно.

**Требования:** Google AI Studio API ключ + Google Drive  
**GPU:** Не нужен  
**Туннель:** Не нужен (автономная работа)  
**Подходит для:** Массовая обработка накопившихся счетов

In [ ]:
!pip install -q google-generativeai

import os
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    print("\u2713 API ключ загружен из Colab Secrets")
except:
    import getpass
    GEMINI_API_KEY = getpass.getpass("Введите Gemini API ключ: ")

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

MODEL_NAME = "gemini-2.5-flash"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Структура папок
INBOX_DIR = "/content/drive/MyDrive/InvoiceLLM/inbox"      # PDF для обработки
PROCESSED_DIR = "/content/drive/MyDrive/InvoiceLLM/processed"  # Результаты JSON
ARCHIVE_DIR = "/content/drive/MyDrive/InvoiceLLM/archive"    # Обработанные PDF

os.makedirs(INBOX_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(ARCHIVE_DIR, exist_ok=True)

# Check for PDFs
pdfs = [f for f in os.listdir(INBOX_DIR) if f.lower().endswith('.pdf')]
print(f"\u2713 Папки созданы в Google Drive/InvoiceLLM/")
print(f"  Найдено PDF в inbox: {len(pdfs)}")
if pdfs:
    for f in pdfs[:10]:
        print(f"    - {f}")
    if len(pdfs) > 10:
        print(f"    ... и ещё {len(pdfs)-10}")
else:
    print("  \u26a0 Положите PDF файлы в Google Drive/InvoiceLLM/inbox/")

In [ ]:
import google.generativeai as genai
import json, time, pathlib, shutil

genai.configure(api_key=GEMINI_API_KEY)

SYSTEM_PROMPT = """You are an invoice data extractor. Analyze the invoice document and extract:
- country: The country of the invoice issuer
- doc_type: One of: electricity, telecom, bank, water, gas, tax, other
- company: The issuing company name
- year: The invoice date year (integer)

Respond ONLY with valid JSON:
{"country": "...", "doc_type": "...", "company": "...", "year": ...}"""

def process_pdf(pdf_path: str) -> dict:
    """Process a single PDF invoice with Gemini multimodal."""
    filename = os.path.basename(pdf_path)
    
    try:
        # Upload PDF to Gemini
        uploaded = genai.upload_file(pdf_path, mime_type="application/pdf")
        
        model = genai.GenerativeModel(
            model_name=MODEL_NAME,
            system_instruction=SYSTEM_PROMPT,
            generation_config=genai.types.GenerationConfig(
                temperature=0.0,
                max_output_tokens=200,
            )
        )
        
        response = model.generate_content([
            "Extract invoice data from this document:",
            uploaded
        ])
        
        # Parse JSON from response
        text = response.text.strip()
        # Remove markdown code blocks if present
        if text.startswith("```"):
            text = text.split("\n", 1)[1].rsplit("```", 1)[0].strip()
        
        result = json.loads(text)
        result["_source"] = filename
        result["_status"] = "ok"
        
        # Cleanup uploaded file
        try:
            genai.delete_file(uploaded.name)
        except:
            pass
            
        return result
        
    except json.JSONDecodeError:
        return {"_source": filename, "_status": "error", "_error": f"Invalid JSON: {text[:200]}"}
    except Exception as e:
        return {"_source": filename, "_status": "error", "_error": str(e)}

# Test with first PDF if available
if pdfs:
    test_pdf = os.path.join(INBOX_DIR, pdfs[0])
    print(f"Тест: {pdfs[0]}")
    result = process_pdf(test_pdf)
    print(json.dumps(result, indent=2, ensure_ascii=False))
else:
    print("Нет PDF для теста. Положите файлы в Google Drive/InvoiceLLM/inbox/")

In [ ]:
import json, time, os, shutil

results = []
errors = []

pdfs = [f for f in os.listdir(INBOX_DIR) if f.lower().endswith('.pdf')]
total = len(pdfs)

if total == 0:
    print("\u26a0 Нет PDF файлов в inbox/")
else:
    print(f"Обработка {total} файлов...\n")
    
    for i, filename in enumerate(pdfs, 1):
        pdf_path = os.path.join(INBOX_DIR, filename)
        print(f"[{i}/{total}] {filename}...", end=" ")
        
        result = process_pdf(pdf_path)
        
        if result.get("_status") == "ok":
            print(f"\u2713 {result.get('country', '?')} / {result.get('doc_type', '?')} / {result.get('company', '?')}")
            results.append(result)
            
            # Save individual JSON
            json_name = os.path.splitext(filename)[0] + ".json"
            json_path = os.path.join(PROCESSED_DIR, json_name)
            with open(json_path, "w", encoding="utf-8") as f:
                json.dump(result, f, indent=2, ensure_ascii=False)
            
            # Move PDF to archive
            shutil.move(pdf_path, os.path.join(ARCHIVE_DIR, filename))
        else:
            print(f"\u2717 {result.get('_error', 'Unknown error')[:60]}")
            errors.append(result)
        
        # Rate limit (Gemini free tier)
        time.sleep(2)
    
    print(f"\n{'='*60}")
    print(f"Готово: {len(results)} успешно, {len(errors)} ошибок")
    print(f"Результаты: Google Drive/InvoiceLLM/processed/")
    print(f"Архив PDF: Google Drive/InvoiceLLM/archive/")

In [ ]:
import json, os

# Load all results
all_results = []
for f in os.listdir(PROCESSED_DIR):
    if f.endswith('.json'):
        with open(os.path.join(PROCESSED_DIR, f), encoding='utf-8') as fp:
            all_results.append(json.load(fp))

if all_results:
    print(f"Всего обработано: {len(all_results)} инвойсов\n")
    
    # Stats by country
    countries = {}
    for r in all_results:
        c = r.get('country', 'Unknown')
        countries[c] = countries.get(c, 0) + 1
    print("По странам:")
    for c, n in sorted(countries.items(), key=lambda x: -x[1]):
        print(f"  {c}: {n}")
    
    # Stats by doc_type
    types = {}
    for r in all_results:
        t = r.get('doc_type', 'unknown')
        types[t] = types.get(t, 0) + 1
    print("\nПо типам:")
    for t, n in sorted(types.items(), key=lambda x: -x[1]):
        print(f"  {t}: {n}")
    
    # Save summary
    summary_path = os.path.join(PROCESSED_DIR, "_summary.json")
    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump({
            "total": len(all_results),
            "by_country": countries,
            "by_type": types,
            "results": all_results
        }, f, indent=2, ensure_ascii=False)
    print(f"\nСводка сохранена: {summary_path}")
else:
    print("Нет обработанных результатов")

## Использование

1. Положите PDF-счета в **Google Drive → InvoiceLLM → inbox/**
2. Запустите все ячейки
3. Результаты JSON в **Google Drive → InvoiceLLM → processed/**
4. Обработанные PDF перемещаются в **archive/**

### Структура Google Drive

```
MyDrive/
└── InvoiceLLM/
    ├── inbox/        ← Положите PDF сюда
    ├── processed/    ← JSON результаты
    └── archive/      ← Обработанные PDF
```

### Multimodal обработка

Gemini обрабатывает PDF напрямую (multimodal) — не нужно извлекать текст. Это работает лучше для сканированных документов и сложных layout-ов.